# MotionGPT: Milestone 2 (pretrained inference in Colab)

Run the cells **top to bottom**. Before starting: **Runtime > Change runtime type > T4 GPU**.

This notebook includes every fix we needed to make the authors' code run on current Colab
(Python 3.13, `transformers` 5.x). See `MILESTONE2.md` for what broke and why.

Repo: https://github.com/OpenMotionLab/MotionGPT  |  Paper: https://arxiv.org/abs/2306.14795

## 1. Environment check (screenshot this)

In [ ]:
import sys, subprocess
print("Python:", sys.version)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch, transformers
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)

## 2. Clone the repo
To use your own fork, replace the URL.

In [ ]:
%cd /content
!git clone https://github.com/OpenMotionLab/MotionGPT.git
%cd /content/MotionGPT

## 3. Install dependencies

`pip install -r requirements.txt` stops at the first package that fails (`chumpy` fails to build here), which leaves
everything else uninstalled. This installs each package separately instead, plus two packages the code needs that are
missing from `requirements.txt` (`bert_score`, `huggingface_hub`). `chumpy` is not needed for the demo.

In [ ]:
import re, subprocess

failed = []
for line in open("/content/MotionGPT/requirements.txt"):
    pkg = line.strip()
    if not pkg or pkg.startswith("#"):
        continue
    r = subprocess.run(["pip", "install", "-q", pkg], capture_output=True, text=True)
    if r.returncode == 0:
        print("OK     ", pkg)
        continue
    name = re.split(r"[=<>~!;@\[ ]", pkg)[0]
    r2 = subprocess.run(["pip", "install", "-q", name], capture_output=True, text=True)
    if r2.returncode == 0:
        print("OK*    ", name, "(unpinned; original was:", pkg + ")")
    else:
        print("FAILED ", pkg)
        failed.append(pkg)
print("\nStill failing:", failed, "(chumpy is expected and not needed)")

In [ ]:
!pip install -q bert_score huggingface_hub
!python -m spacy download en_core_web_sm

## 4. Download dependencies

- **Evaluator files** (mean/std statistics and word vectors) come from the repo's script. They extract one folder
  deeper than the code expects, so the next cell copies them into place.
- **T5 model and MotionGPT checkpoint** come from Hugging Face with `huggingface_hub`. The repo's own scripts use
  `git clone` with `git lfs`, which silently saves tiny stub files when git-lfs is missing.

In [ ]:
%cd /content/MotionGPT
!bash prepare/download_t2m_evaluators.sh

In [ ]:
import os, shutil
os.chdir("/content/MotionGPT")

# Fix 1: evaluator files extracted one folder too deep
nested, target = "deps/t2m/t2m/t2m", "deps/t2m/t2m"
if os.path.exists(nested):
    for name in os.listdir(nested):
        src, dst = os.path.join(nested, name), os.path.join(target, name)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy(src, dst)

# Fix 2: word vectors must be at deps/glove
if os.path.exists("deps/t2m/glove") and not os.path.exists("deps/glove/our_vab_data.npy"):
    shutil.copytree("deps/t2m/glove", "deps/glove", dirs_exist_ok=True)

print("mean.npy:", os.path.exists("deps/t2m/t2m/VQVAEV3_CB1024_CMT_H1024_NRES3/meta/mean.npy"))
print("eval mean.npy:", os.path.exists("deps/t2m/t2m/Comp_v6_KLD01/meta/mean.npy"))
print("glove:", os.path.exists("deps/glove/our_vab_data.npy"))

In [ ]:
import os, shutil
from huggingface_hub import snapshot_download
os.chdir("/content/MotionGPT")

def big_enough(path, mb):
    return os.path.exists(path) and os.path.getsize(path) > mb * 1e6

if not big_enough("deps/flan-t5-base/model.safetensors", 100):
    shutil.rmtree("deps/flan-t5-base", ignore_errors=True)
    snapshot_download("google/flan-t5-base", local_dir="deps/flan-t5-base",
                      allow_patterns=["*.json", "spiece.model", "model.safetensors"])

if not big_enough("checkpoints/MotionGPT-base/motiongpt_s3_h3d.tar", 100):
    shutil.rmtree("checkpoints/MotionGPT-base", ignore_errors=True)
    snapshot_download("OpenMotionLab/MotionGPT-base", local_dir="checkpoints/MotionGPT-base")

for p in ["deps/flan-t5-base/model.safetensors", "checkpoints/MotionGPT-base/motiongpt_s3_h3d.tar"]:
    print(f"{os.path.getsize(p)/1e6:8.1f} MB  {p}")

## 5. Patch the repo code for current library versions

Each patch is idempotent (safe to run twice). The first two are required; the rest are defensive.

1. **`mGPT/data/HumanML3D.py`**: the demo does not need the full HumanML3D dataset, which is not downloaded. The code
   only used it to read the feature count (263), so that value is set directly.
2. **`demo.py`**: newer `transformers` ties the output layer (`lm_head`) to the input embeddings, so loading the
   checkpoint overwrote one with the other and the model produced noise (4-frame outputs). The patch restores both
   from the checkpoint.
3. **`demo.py`**: switch the model to eval mode, and pass `weights_only=False` to `torch.load` (newer PyTorch blocks
   this by default; the file is the authors' official checkpoint).
4. **`mGPT/archs/mgpt_lm.py`**: pass the attention mask to `generate`.

In [ ]:
import os
os.chdir("/content/MotionGPT")

# 1. Skip loading the full HumanML3D dataset
p = "mGPT/data/HumanML3D.py"
s = open(p).read()
old = '''        self._sample_set = self.get_sample_set(overrides={"split": "test", "tiny": True})
        self.nfeats = self._sample_set.nfeats
        cfg.DATASET.NFEATS = self.nfeats'''
new = '''        # Demo patch: skip loading the full HumanML3D dataset (not downloaded).
        # HumanML3D motion features are always 263-dimensional.
        self.nfeats = 263
        cfg.DATASET.NFEATS = self.nfeats'''
if old in s:
    open(p, "w").write(s.replace(old, new))
print("HumanML3D.py patched:", "Demo patch" in open(p).read())

# 2 + 3. demo.py: weights_only, eval mode, untie lm_head
s = open("demo.py").read()
if "Colab patch" not in s:
    s = s.replace('map_location="cpu")["state_dict"]', 'map_location="cpu", weights_only=False)["state_dict"]')
    s = s.replace("    model.to(device)\n", "    model.to(device)\n    model.eval()\n", 1)
    s = s.replace(
        "        model.load_state_dict(state_dict)\n",
        '''        model.load_state_dict(state_dict)
        # --- Colab patch: newer transformers may tie lm_head to the input embeddings ---
        _lm = model.lm.language_model
        print("DIAG lm_head tied to shared:", _lm.lm_head.weight.data_ptr() == _lm.shared.weight.data_ptr())
        _ck_shared = state_dict.get("lm.language_model.shared.weight")
        _ck_head = state_dict.get("lm.language_model.lm_head.weight")
        if _ck_shared is not None and _ck_head is not None:
            if _lm.lm_head.weight.data_ptr() == _lm.shared.weight.data_ptr():
                _lm.config.tie_word_embeddings = False
                _lm.lm_head.weight = torch.nn.Parameter(_lm.lm_head.weight.detach().clone())
            with torch.no_grad():
                _lm.shared.weight.copy_(_ck_shared)
                _lm.lm_head.weight.copy_(_ck_head)
''', 1)
    open("demo.py", "w").write(s)
print("demo.py patched:", "Colab patch" in open("demo.py").read())

# 4. mgpt_lm.py: attention mask
p = "mGPT/archs/mgpt_lm.py"
s = open(p).read()
old = "                source_input_ids,\n                max_length=max_length,"
new = "                source_input_ids,\n                attention_mask=source_attention_mask,\n                max_length=max_length,"
if old in s:
    open(p, "w").write(s.replace(old, new, 1))
print("mgpt_lm.py patched:", "attention_mask=source_attention_mask,\n                max_length" in open(p).read())

## 6. Prompts

The model was instruction-tuned, so prompts work best as instructions that start with "Generate ...".

In [ ]:
prompts = [
    "Generate a person walks forward and then turns around.",
    "Generate a person waves with their right hand.",
    "Generate a person jumps up and down.",
    "Generate a person walks in a circle.",
    "Generate a person kicks with their left leg.",
]
with open("/content/MotionGPT/demos/my_prompts.txt", "w") as f:
    f.write("\n".join(prompts) + "\n")
print(open("/content/MotionGPT/demos/my_prompts.txt").read())

## 7. Run inference: text-to-motion (the core Milestone 2 result)
Screenshot the output of this cell.

In [ ]:
%%time
%cd /content/MotionGPT
!python demo.py --cfg ./configs/config_h3d_stage3.yaml --example ./demos/my_prompts.txt

## 8. Locate the generated motions

Motions are saved as `<number>_out.npy` with shape `(1, frames, 22, 3)`. The `_in.npy` files are empty placeholders.
Motion length is generated by the model (the repo's config caps it at 196 frames, about 10 seconds at 20 fps).

In [ ]:
import os, glob
import numpy as np
os.chdir("/content/MotionGPT")

latest = sorted(glob.glob("/content/MotionGPT/results/mgpt/*/samples_*"))[-1]
found = sorted(glob.glob(latest + "/*_out.npy"), key=lambda p: int(os.path.basename(p).split("_")[0]))
print("folder:", latest)
for i, p in enumerate(found):
    shape = np.load(p).shape
    print(os.path.basename(p), shape, f"({shape[1] / 20:.1f} s at 20 fps)")
print("\nraw output, prompt 0 (should be '<Motion_Placeholder>'):")
print(repr(open(f"{latest}/0_out.txt").read())[:120])

## 9. Skeleton animation (matplotlib, no Blender needed)
Turns each `.npy` into a GIF for your slides. This takes a few minutes.

In [ ]:
import numpy as np, os
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

# HumanML3D 22-joint skeleton chains
CHAINS = [[0, 2, 5, 8, 11], [0, 1, 4, 7, 10], [0, 3, 6, 9, 12, 15],
          [9, 14, 17, 19, 21], [9, 13, 16, 18, 20]]
COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

def animate(npy_path, out_path, step=2, title=""):
    d = np.load(npy_path)
    if d.ndim == 4:
        d = d[0]
    n = d.shape[0]
    mins, maxs = d.reshape(-1, 3).min(0), d.reshape(-1, 3).max(0)
    center = (mins + maxs) / 2
    r = (maxs - mins).max() / 2 + 0.3
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(111, projection="3d")
    frames = list(range(0, n, step))      # every 2nd frame keeps the GIF small and fast to render

    def update(i):
        ax.cla()
        ax.set_xlim(center[0] - r, center[0] + r)
        ax.set_ylim(center[2] - r, center[2] + r)
        ax.set_zlim(center[1] - r, center[1] + r)
        ax.view_init(elev=15, azim=-70)
        ax.set_title(f"{title}\nframe {i + 1}/{n}", fontsize=8)
        for chain, c in zip(CHAINS, COLORS):
            ax.plot(d[i, chain, 0], d[i, chain, 2], d[i, chain, 1], color=c, linewidth=3, marker="o", markersize=3)

    anim = FuncAnimation(fig, update, frames=frames)
    anim.save(out_path, writer=PillowWriter(fps=int(20 / step)))
    plt.close(fig)

os.makedirs("/content/animations", exist_ok=True)
gifs = []
for i, p in enumerate(found):
    label = prompts[i].replace("Generate ", "") if i < len(prompts) else os.path.basename(p)
    out = f"/content/animations/motion_{i}.gif"
    animate(p, out, title=label)
    gifs.append(out)
    print(label, "->", out)

for g in gifs:
    display(Image(filename=g))

## 10. (Optional) Other tasks: m2t, pred, inbetween

The repo ships example inputs in `demos/` (`m2t.txt`, `pred.txt`, `inbetween.txt`). These were **not tested** for this
milestone. Their lines point at motion feature files, so open each file first to check what it expects. Uncomment to try.

In [ ]:
# !head -c 600 demos/m2t.txt
# !python demo.py --cfg ./configs/config_h3d_stage3.yaml --task m2t --example ./demos/m2t.txt

## 11. Save your evidence
Zips the animations, motion files and prompts, and downloads them (also copies to Drive if mounted).

In [ ]:
import shutil, os
os.chdir("/content/MotionGPT")
os.makedirs("/content/evidence", exist_ok=True)
for p in found:
    shutil.copy(p, "/content/evidence/")
shutil.copy("demos/my_prompts.txt", "/content/evidence/")
shutil.copytree("/content/animations", "/content/evidence/animations", dirs_exist_ok=True)
shutil.make_archive("/content/motiongpt_evidence", "zip", "/content/evidence")

from google.colab import files
files.download("/content/motiongpt_evidence.zip")

if os.path.exists("/content/drive/MyDrive"):
    shutil.copy("/content/motiongpt_evidence.zip", "/content/drive/MyDrive/")
    print("Also copied to Google Drive.")

## Slide checklist

- Environment screenshot (section 1)
- The challenges and fixes (see `MILESTONE2.md`)
- Demo run output with timing (section 7) and the file shapes (section 8)
- 3 to 5 GIFs (section 9)
- Repo link and README with these exact steps